In [3]:

import requests, re, json
from bs4 import BeautifulSoup

WEB_HDR = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0'}

def parse_bechcicki_product(pid_str):
    """pid_str = '0026479' (cyfry bez P-)"""
    url = f'https://www.bechcicki.pl/{pid_str}-id-p-{pid_str}'
    r = requests.get(url, headers=WEB_HDR, timeout=20, allow_redirects=True)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, 'html.parser')

    # Wyciągnij __NUXT__ JSON
    nuxt_script = None
    for s in soup.find_all('script'):
        t = s.string or ''
        if '__NUXT__' in t:
            nuxt_script = t
            break
    if not nuxt_script:
        return None

    # Wyciągnij JSON z window.__NUXT__=...
    m = re.search(r'window\.__NUXT__\s*=\s*(\{.*\})\s*;?\s*$', nuxt_script, re.DOTALL)
    if not m:
        # Spróbuj inaczej - weź całe dane po '='
        m = re.search(r'__NUXT__\s*=\s*(.+)', nuxt_script, re.DOTALL)
    if not m:
        return None

    try:
        data = json.loads(m.group(1).rstrip(';'))
    except:
        return None

    # Znajdź produkt w state
    def find_key(obj, key, depth=0):
        if depth > 8: return None
        if isinstance(obj, dict):
            if key in obj: return obj[key]
            for v in obj.values():
                r = find_key(v, key, depth+1)
                if r is not None: return r
        elif isinstance(obj, list):
            for v in obj:
                r = find_key(v, key, depth+1)
                if r is not None: return r
        return None

    product = find_key(data, 'product')
    return product, data

result = parse_bechcicki_product('0026479')
if result:
    product, data = result
    print("Klucze w product:", sorted(product.keys()) if isinstance(product, dict) else type(product))
    if isinstance(product, dict):
        print(json.dumps({k: v for k,v in product.items() if k not in ['description','longDescription']}, 
                         indent=2, ensure_ascii=False)[:2000])


In [7]:

import requests, re, json
from bs4 import BeautifulSoup

WEB_HDR = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0'}

pid_str = '0026479'
url = f'https://www.bechcicki.pl/{pid_str}-id-p-{pid_str}'
r = requests.get(url, headers=WEB_HDR, timeout=20, allow_redirects=True)
soup = BeautifulSoup(r.text, 'html.parser')

# Pobierz __NUXT__ script
nuxt_script = ''
for s in soup.find_all('script'):
    t = s.string or ''
    if '__NUXT__' in t:
        nuxt_script = t
        break

print(f"Długość: {len(nuxt_script)}")
print("Pierwsze 500 znaków:")
print(nuxt_script[:500])
print("\n... \nOstatnie 200 znaków:")
print(nuxt_script[-200:])


Długość: 830863
Pierwsze 500 znaków:
window.__NUXT__=(function(a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p,q,r,s,t,u,v,w,x,y,z,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R,S,T,U,V,W,X,Y,Z,_,$,aa,ab,ac,ad,ae,af,ag,ah,ai,aj,ak,al,am,an,ao,ap,aq,ar,as,at,au,av,aw,ax,ay,az,aA,aB,aC,aD,aE,aF,aG,aH,aI,aJ,aK,aL,aM,aN,aO,aP,aQ,aR,aS,aT,aU,aV,aW,aX,aY,aZ,a_,a$,ba,bb,bc,bd,be,bf,bg,bh,bi,bj,bk,bl,bm,bn,bo,bp,bq,br,bs,bt,bu,bv,bw,bx,by,bz,bA,bB,bC,bD,bE,bF,bG,bH,bI,bJ,bK,bL,bM,bN,bO,bP,bQ,bR,bS,bT,bU,bV,bW,bX,bY,bZ,b_,b$,ca,cb,cc,cd,ce,cf,cg,ch,ci,cj,ck,cl,cm,cn,

... 
Ostatnie 200 znaków:
ać?","Produkt kablowy wymaga ponownej konfiguracji, nie może zostać dodany do koszyka.","Towar wystawiony notami","Przedsprzedaż","Nie znaleziono żadnych produktów.","Dodaj","Brak dostępnych opcji"));


In [11]:

import requests, re, json
from bs4 import BeautifulSoup

WEB_HDR = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}
pid_str = '0026479'
r = requests.get(f'https://www.bechcicki.pl/{pid_str}-id-p-{pid_str}',
                 headers=WEB_HDR, timeout=20, allow_redirects=True)
soup = BeautifulSoup(r.text, 'html.parser')

# 1. JSON-LD
for s in soup.find_all('script', type='application/ld+json'):
    try:
        ld = json.loads(s.string)
        print("JSON-LD:", json.dumps(ld, ensure_ascii=False)[:600])
    except: pass

# 2. Breadcrumb prawdziwy
bcs = soup.select('nav ol li a, ol.breadcrumb li a, [itemtype*=BreadcrumbList] a')
print("\nBreadcrumb links:", [(b.text.strip(), b.get('href','')[:40]) for b in bcs])

# 3. Szukaj sekcji parametrów
param_section = soup.select_one('#parametersList, [id*=param], [class*=param-list], section.params')
if param_section:
    print("\nParam section found:", param_section.name, param_section.get('class'))
    print(param_section.text[:400])
else:
    # Poszukaj "Parametry techniczne" nagłówka i weź sibling
    for h in soup.find_all(['h2','h3','h4','div'], string=re.compile('Parametry', re.I)):
        print("\nHeader:", h.text[:50])
        nxt = h.find_next_sibling()
        if nxt: print("Sibling:", nxt.text[:200])


JSON-LD: {"@context": "https://schema.org", "@type": "BreadcrumbList", "itemListElement": [{"@type": "ListItem", "position": 1, "name": "Chemia budowlana", "item": "/chemia-budowlana"}, {"@type": "ListItem", "position": 2, "name": "Gipsy i gładzie", "item": "/chemia-budowlana/gipsy-i-gladzie"}, {"@type": "ListItem", "position": 3, "name": "Gładzie masy gotowe", "item": "/chemia-budowlana/gipsy-i-gladzie/gladzie-masy-gotowe"}, {"@type": "ListItem", "position": 4, "name": "Gładź gipsowa Knauf G-K Finish 25 kg"}]}
JSON-LD: {"@context": "https://schema.org", "@type": "Product", "additionalProperty": [{"@type": "PropertyValue", "unitText": "Ilość na palecie", "value": 48}, {"@type": "PropertyValue", "unitText": "Rodzaj gładzi/gipsu", "value": "Gładź gipsowa"}, {"@type": "PropertyValue", "unitText": "Wydajność", "value": "1 kg/m²"}, {"@type": "PropertyValue", "unitText": "Metoda aplikacji", "value": "ręczna"}, {"@type": "PropertyValue", "unitText": "Zastosowanie", "value": "wewnętrzne"}, {"@